# 10 · Son Model ve İyileştirmeler

Elle değerlendirme (09) ve önceki deneylerde bulunan eksikler burada kapatılır. Sonuçlar demoya (05) aktarılır.

| Bölüm | Eksik | Yapılan |
|---|---|---|
| A | Tablo özelliği 04'te denendi ama son modelde yoktu | Karakter TF-IDF + tablo özellikleri tek modelde, olasılık veren (kalibre) Linear SVM |
| B | Model kararsızken bile bir tür söylüyor; hataların çoğu "fatura" tahmini | **"Emin değilim"** çıktısı: eşik validation setinde seçilir |
| C | İçerik çıkarımında hatalar (precision 0,52) | Kurallar düzeltildi; aynı 20 belgede önce/sonra karşılaştırması |

**Veri kuralı:** Model, ağırlık ve eşik seçimi sadece train + validation ile yapılır; test seti en sonda bir kez kullanılır.

**Girdi:** `data/cleaned_dataset.csv` (03), `data/table_features.csv` (04; eksikler burada hesaplanır) · **Çıktı:** `models/final_model.joblib`, `models/final_model.json` (05 demo bunları kullanır). Süre ~5–8 dk, GPU gerekmez.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract opencv-python-headless spacy symspellpy
!python -m spacy download en_core_web_sm -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2, joblib
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA, MODELS = ROOT / "data", ROOT / "models"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
CFG_FILE = ROOT / "siniflar.json"
CLASSES = json.loads(CFG_FILE.read_text())["CLASSES"] if CFG_FILE.exists() else ["email", "form", "invoice", "resume"]
SEED = 42
TR = {"email": "E-posta", "invoice": "Fatura", "resume": "CV", "scientific_publication": "Bilimsel makale",
      "form": "Form", "letter": "Mektup", "news_article": "Gazete", "emin_degilim": "Emin değilim"}
print("Sınıflar:", CLASSES)

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/final_model.py
"""Son model: karakter TF-IDF + tablo özellikleri → olasılık veren Linear SVM (Platt kalibrasyonu).

04'teki ek deneyde tablo özelliği (asistan önerisi) Linear SVM'in test başarısını artırdı. Bu modül o
deneyi demoda kullanılabilecek tek bir modele dönüştürür:

    metin (normalize_for_model çıktısı) ──► karakter TF-IDF ─┐
                                                              ├─► Linear SVM (kalibre) ─► olasılıklar
    görüntü ──► detect_table ──► 5 tablo özelliği (ölçekli) ─┘

"Emin değilim": en yüksek olasılık `threshold`'un altındaysa model karar vermez; en olası iki türü
gösterir. Eşik validation setinde seçilir (test setine bakılmaz).
"""
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

TAB_COLS = ["n_hlines", "n_vlines", "n_intersections", "line_ratio", "has_table"]
UNSURE = "emin_degilim"


def char_tfidf():
    # 04'teki karakter TF-IDF ile aynı ayarlar
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9,
                           sublinear_tf=True, max_features=100_000)


def tab_array(rows):
    """detect_table sözlükleri ya da TAB_COLS sütunlu DataFrame → (n, 5) dizi; sayılar log ölçekte."""
    if hasattr(rows, "loc"):
        t = rows[TAB_COLS].astype(float).values.copy()
    else:
        t = np.array([[float(r[c]) for c in TAB_COLS] for r in rows])
    t[:, :3] = np.log1p(t[:, :3])
    return t


class TextTableModel:
    def __init__(self, C=0.5, table_weight=0.2, threshold=0.0, seed=42):
        self.C, self.table_weight, self.threshold, self.seed = C, table_weight, threshold, seed

    def _X(self, texts, tabs, fit=False):
        if fit:
            self.vec_ = char_tfidf().fit(texts)
            self.scaler_ = StandardScaler().fit(tab_array(tabs))
        X_text = self.vec_.transform(texts)
        if self.table_weight == 0:
            return X_text.tocsr()
        X_tab = csr_matrix(self.table_weight * self.scaler_.transform(tab_array(tabs)))
        return hstack([X_text, X_tab]).tocsr()

    def fit(self, texts, tabs, y, calibrate=True):
        X = self._X(texts, tabs, fit=True)
        svm = LinearSVC(C=self.C, random_state=self.seed)
        self.clf_ = CalibratedClassifierCV(svm, cv=5, method="sigmoid").fit(X, y) if calibrate else svm.fit(X, y)
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, texts, tabs):
        return self.clf_.predict_proba(self._X(texts, tabs))

    def predict(self, texts, tabs):
        return self.clf_.predict(self._X(texts, tabs))

    def decide(self, texts, tabs, threshold=None):
        """Tahmin + 'emin değilim': en yüksek olasılık eşiğin altındaysa UNSURE döner."""
        t = self.threshold if threshold is None else threshold
        P = self.predict_proba(texts, tabs)
        top = P.argmax(axis=1)
        labels = self.classes_[top].astype(object)
        labels[P.max(axis=1) < t] = UNSURE
        return labels, P


def coverage_table(y_true, P, classes, thresholds):
    """Her eşik için: kapsama (karar verilen belge oranı), karar verilenlerdeki doğruluk, kaçan hata sayısı."""
    y_true = np.asarray(y_true)
    pred = np.asarray(classes)[P.argmax(axis=1)]
    conf = P.max(axis=1)
    rows = []
    for t in thresholds:
        keep = conf >= t
        n_err_all = int((pred != y_true).sum())
        n_err_kept = int((pred[keep] != y_true[keep]).sum())
        rows.append({
            "esik": t,
            "kapsama": keep.mean(),
            "karar_verilenlerde_dogruluk": (pred[keep] == y_true[keep]).mean() if keep.any() else np.nan,
            "emin_degilim_sayisi": int((~keep).sum()),
            "yakalanan_hata": n_err_all - n_err_kept,
            "kalan_hata": n_err_kept,
            "gereksiz_kararsizlik": int(((~keep) & (pred == y_true)).sum()),
        })
    return rows


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_core.py
import cv2
import numpy as np
import pytesseract


def estimate_skew(gray, max_angle=5.0, step=0.25):
    """Projection-profile skew estimate: the angle whose row sums are sharpest."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = cv2.medianBlur(th, 3)  # drop isolated noise pixels
    h, w = th.shape
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + 1e-9, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        r = cv2.warpAffine(th, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(r.sum(axis=1, dtype=np.float64)))
        if score > best_score:
            best_angle, best_score = float(a), score
    return best_angle


def deskew(gray):
    """Rotate the page so text lines are horizontal."""
    angle = estimate_skew(gray)
    if abs(angle) < 0.3:
        return gray, 0.0
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def preprocess(img_bgr_or_gray, upscale_to=2000, do_deskew=True, binarize=True):
    """Grayscale -> upscale (RVL-CDIP is max 1000px) -> denoise -> deskew -> Otsu."""
    img = img_bgr_or_gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    gray = cv2.medianBlur(gray, 3)  # salt-and-pepper scan noise, before upscaling
    h, w = gray.shape
    scale = upscale_to / max(h, w)
    if scale > 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    angle = 0.0
    if do_deskew:
        gray, angle = deskew(gray)
    if binarize:
        _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, angle


def run_ocr(img, lang="eng", psm=3, min_conf=0):
    """Tesseract OCR -> words, normalized boxes (0-1000, LayoutLM format), line text."""
    h, w = img.shape[:2]
    d = pytesseract.image_to_data(img, lang=lang, config=f"--oem 1 --psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    words, boxes, confs, lines = [], [], [], {}
    for i, txt in enumerate(d["text"]):
        txt = txt.strip()
        conf = float(d["conf"][i])
        if not txt or conf < min_conf:
            continue
        x, y, bw, bh = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        words.append(txt)
        confs.append(conf)
        boxes.append([int(1000 * x / w), int(1000 * y / h),
                      int(1000 * (x + bw) / w), int(1000 * (y + bh) / h)])
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(txt)
    text = "\n".join(" ".join(v) for _, v in sorted(lines.items()))
    return {
        "text": text,
        "words": words,
        "boxes": boxes,
        "confs": confs,
        "n_words": len(words),
        "mean_conf": float(np.mean(confs)) if confs else 0.0,
    }


def detect_table(gray):
    """Morfolojik işlemlerle yatay/dikey çizgileri bulup belgede tablo olup olmadığını tahmin eder.

    OCR tabloları bozarak okuduğu için (danışman/asistan görüşmesi) tablo bilgisi
    görüntüden ayrıca çıkarılır ve sınıflandırmada ek özellik olarak kullanılabilir.
    """
    img = gray if gray.ndim == 2 else cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    h, w = img.shape
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(10, w // 25), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(10, h // 25)))
    horiz = cv2.morphologyEx(th, cv2.MORPH_OPEN, hk)
    vert = cv2.morphologyEx(th, cv2.MORPH_OPEN, vk)
    n_h = len(cv2.findContours(horiz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    n_v = len(cv2.findContours(vert, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    inter = cv2.bitwise_and(cv2.dilate(horiz, np.ones((3, 3), np.uint8)), cv2.dilate(vert, np.ones((3, 3), np.uint8)))
    n_x = len(cv2.findContours(inter, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    line_ratio = float((horiz > 0).mean() + (vert > 0).mean())
    return {
        "n_hlines": n_h,
        "n_vlines": n_v,
        "n_intersections": n_x,
        "line_ratio": round(line_ratio, 5),
        "has_table": bool(n_h >= 3 and n_v >= 2 and n_x >= 4),
    }


def rotate_bound(gray, angle):
    """Görüntüyü kırpmadan döndürür (köşeler beyazla doldurulur)."""
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(gray, M, (nw, nh), flags=cv2.INTER_CUBIC, borderValue=255)


def _good_words(o, min_conf=70):
    return sum(c >= min_conf for c in o["confs"])


def upscale_if_small(gray, ocr, target_px=24):
    """Yazı küçükse görüntüyü büyütüp yeniden okur (21 Ekim).

    Tesseract harf yüksekliği ~20-30 piksel olduğunda en iyi çalışır. RVL-CDIP taramalarında kelime yüksekliği
    7-12 piksel; internetten alınan küçük resimlerde daha da az. Kelime kutularının ortanca yüksekliği
    hedefin altındaysa görüntü büyütülür; güvenli (>= 70) okunan kelime sayısı artarsa büyütülmüş okuma seçilir.
    """
    hs = [b[3] - b[1] for b in ocr["boxes"]]
    if not hs:
        med = 0.0
    else:
        med = float(np.median(hs)) * gray.shape[0] / 1000.0
    if med >= target_px * 0.8 and _good_words(ocr) > 0:
        return gray, ocr, 1.0
    f = 2.0   # denemelerde x2 en dengelisi: x3 bazı tablo sütunlarını kaybettirdi
    big = cv2.resize(gray, None, fx=f, fy=f, interpolation=cv2.INTER_CUBIC)
    o2 = run_ocr(big)
    return (big, o2, f) if _good_words(o2) > 1.1 * _good_words(ocr) else (gray, ocr, 1.0)


def ocr_with_orientation(img, upscale=True):
    """Yan, eğik ya da ters dönmüş sayfayı düzeltip OCR yapar.

    21 Ekim: elle değerlendirmede ~77° dönük bir fatura hiç okunamamıştı ("WoRsN"). Yazı satırlarının açısı,
    satır toplamlarının en keskin olduğu açı aranarak (-90°…+90°) bulunur; satır yönü 180° belirsiz olduğu için
    sayfa o açıyla ve ters yönüyle okunur. Açı küçükse Tesseract'ın yön tespiti (OSD) baş aşağı sayfaya bakar.
    Aday hâller arasından en çok ve en emin kelimeyi veren seçilir; asıl hâlden en az %20 iyi değilse sayfaya
    dokunulmaz, böylece düz belgeler bozulmaz.
    Dönüş: (görüntü, ocr sonucu, döndürme açısı)
    """
    import re
    gray = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ocr = run_ocr(gray)
    score = lambda o: o["n_words"] * o["mean_conf"]
    angles = []
    a = estimate_skew(gray, max_angle=90, step=1.0)
    if abs(a) >= 3:
        angles += [a, a + 180]
    else:
        try:
            osd = pytesseract.image_to_osd(gray, config="--psm 0 -c min_characters_to_try=5")
            if int(re.search(r"Rotate: (\d+)", osd).group(1)) == 180:
                angles.append(180)
        except Exception:
            pass
    best = (gray, ocr, 0)
    for ang in angles:
        r = rotate_bound(gray, ang)
        o = run_ocr(r)
        if score(o) > 1.2 * score(best[1]):
            best = (r, o, ang)
    if upscale:
        g2, o2, _ = upscale_if_small(best[0], best[1])
        best = (g2, o2, best[2])
    return best


In [ ]:
sys.path.insert(0, str(ROOT))
import final_model, ocr_core
for m in (final_model, ocr_core): importlib.reload(m)
from final_model import TextTableModel, coverage_table, UNSURE, TAB_COLS
from ocr_core import detect_table

df = pd.read_csv(DATA / "cleaned_dataset.csv")
df = df[df["label"].isin(CLASSES)].reset_index(drop=True)
df["text_norm"] = df["text_norm"].fillna("")

# Tablo özellikleri: 04'te hesaplananlar önbellekten, eksikler burada
TAB_FILE = DATA / "table_features.csv"
tab = pd.read_csv(TAB_FILE) if TAB_FILE.exists() else pd.DataFrame(columns=["path"] + TAB_COLS)
todo = sorted(set(df["path"]) - set(tab["path"]))
if todo:
    from tqdm.auto import tqdm
    new = [{"path": p, **detect_table(cv2.imread(str(DATA / p), cv2.IMREAD_GRAYSCALE))} for p in tqdm(todo)]
    tab = pd.concat([tab, pd.DataFrame(new)], ignore_index=True)
    try: tab.to_csv(TAB_FILE, index=False)
    except OSError as e: print("Kaydedilemedi:", e)
df = df.merge(tab[["path"] + TAB_COLS], on="path", how="left")
assert df[TAB_COLS].notna().all().all(), "Tablo özelliği eksik belge var"

train = df[(df.split == "train") & (~df.is_empty)]
val = df[df.split == "validation"]
test = df[df.split == "test"]
print(f"Train {len(train)} · Validation {len(val)} · Test {len(test)}")

## A. Son model: metin + tablo, olasılık veren

04'teki tablo deneyinde Linear SVM'in C değeri ve tablo ağırlığı ayrı ayrı seçilmişti. Burada ikisi birlikte, **validation macro-F1** ile seçilir.
Sonra seçilen model **olasılık kalibrasyonu** (Platt, 5 katlı) ile eğitilir: demoda "%92 CV" diyebilmek ve "emin değilim" eşiği koyabilmek için olasılık gerekiyor.

In [ ]:
grid = []
for C in [0.1, 0.5, 1.0]:
    for w in [0.0, 0.1, 0.2, 0.5, 1.0]:
        m = TextTableModel(C=C, table_weight=w, seed=SEED).fit(train.text_norm, train, train.label, calibrate=False)
        grid.append({"C": C, "tablo_agirligi": w,
                     "val_macro_f1": f1_score(val.label, m.predict(val.text_norm, val), average="macro")})
grid = pd.DataFrame(grid).sort_values("val_macro_f1", ascending=False)
display(grid.head(8).round(4))
C_best, w_best = float(grid.iloc[0].C), float(grid.iloc[0].tablo_agirligi)
print(f"Seçilen: C = {C_best}, tablo ağırlığı = {w_best}")

In [ ]:
final = TextTableModel(C=C_best, table_weight=w_best, seed=SEED).fit(train.text_norm, train, train.label)
text_only = TextTableModel(C=C_best, table_weight=0.0, seed=SEED).fit(train.text_norm, train, train.label)
old = joblib.load(MODELS / "best_model.joblib")           # 04'ün seçtiği model (sadece metin)
old_info = json.loads((MODELS / "best_model.json").read_text())

preds = {f"04 modeli ({old_info['model']}, {old_info['özellik']})": old.predict(test.text_norm),
         "Karakter TF-IDF · sadece metin (kalibre)": text_only.predict(test.text_norm, test),
         "Karakter TF-IDF + tablo (kalibre) · SON MODEL": final.predict(test.text_norm, test)}
rows = []
for name, p in preds.items():
    f1s = f1_score(test.label, p, labels=CLASSES, average=None)
    rows.append({"model": name, "accuracy": accuracy_score(test.label, p), "macro_f1": f1_score(test.label, p, average="macro"),
                 **{f"F1 {TR.get(c, c)}": v for c, v in zip(CLASSES, f1s)}})
comp = pd.DataFrame(rows).round(4)
display(comp)
comp.to_csv(ROOT / "son_model_karsilastirma.csv", index=False)

p_old, p_new = list(preds.values())[0], list(preds.values())[2]
a, b = (p_old == test.label.values), (p_new == test.label.values)
n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
print(f"McNemar (04 modeli vs son model): sadece 04 doğru {n01}, sadece son model doğru {n10}, "
      f"p = {binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0:.4f}")

**Yorum:** p > 0,05 ise fark istatistiksel olarak anlamlı değildir (test seti küçük); yine de tablo özelliği faturada görüntüden gelen ek bir kanıt sağlar ve sonraki adımlarda (form sınıfı eklendiğinde) daha önemli olabilir.

## B. "Emin değilim" çıktısı

Model kararsızsa (en yüksek olasılık eşiğin altındaysa) tür söylemek yerine **"emin değilim — büyük ihtimalle X ya da Y"** der.
Eşik yükseldikçe daha az belgeye karar verilir (kapsama düşer) ama verilen kararlar daha doğru olur.

**Eşik seçimi (validation):** Her eşik için *yakalanan hata* (yanlış tahminin "emin değilim"e dönmesi) ile *gereksiz kararsızlık* (doğru tahminin "emin değilim"e dönmesi) sayılır.
Farkı (yakalanan − gereksiz) en büyük olan eşik seçilir: bir hatayı yakalamak için en fazla bir doğru tahmin feda edilir. Eşitlikte düşük eşik seçilir.

In [ ]:
TH = np.round(np.arange(0.30, 0.96, 0.05), 2)
P_val = final.predict_proba(val.text_norm, val)
cov_val = pd.DataFrame(coverage_table(val.label, P_val, final.classes_, TH))
cov_val["net"] = cov_val.yakalanan_hata - cov_val.gereksiz_kararsizlik
threshold = float(cov_val.sort_values(["net", "esik"], ascending=[False, True]).iloc[0].esik)
final.threshold = threshold
display(cov_val.round(3))
print(f"Seçilen eşik: {threshold}")

P_te = final.predict_proba(test.text_norm, test)
cov_te = pd.DataFrame(coverage_table(test.label, P_te, final.classes_, TH))
cov_te.round(4).to_csv(ROOT / "emin_degilim_esik_tablosu.csv", index=False)
r = cov_te[cov_te.esik == threshold].iloc[0]
print(f"\nTEST (eşik {threshold}): {int(r.emin_degilim_sayisi)} belgede 'emin değilim' (%{100 * (1 - r.kapsama):.1f}), "
      f"karar verilenlerde doğruluk %{100 * r.karar_verilenlerde_dogruluk:.2f}; "
      f"{int(r.yakalanan_hata)} hata 'emin değilim'e dönüştü, {int(r.kalan_hata)} hata kaldı, "
      f"{int(r.gereksiz_kararsizlik)} doğru tahmin gereksiz yere 'emin değilim' oldu.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(cov_val.kapsama * 100, cov_val.karar_verilenlerde_dogruluk * 100, "o-", label="validation")
ax[0].plot(cov_te.kapsama * 100, cov_te.karar_verilenlerde_dogruluk * 100, "s--", label="test")
for _, rr in cov_te.iterrows():
    if rr.esik in (0.5, threshold, 0.9):
        ax[0].annotate(f"eşik {rr.esik}", (rr.kapsama * 100, rr.karar_verilenlerde_dogruluk * 100), fontsize=8,
                       xytext=(4, -10), textcoords="offset points")
ax[0].set_xlabel("Karar verilen belge (%)"); ax[0].set_ylabel("Karar verilenlerde doğruluk (%)")
ax[0].set_title("Kapsama – doğruluk dengesi"); ax[0].grid(alpha=.3); ax[0].legend()

labels, _ = final.decide(test.text_norm, test)
order = CLASSES + [UNSURE]
cm = confusion_matrix(test.label, labels, labels=order)[:len(CLASSES)]
ax[1].imshow(cm, cmap="Blues")
ax[1].set_xticks(range(len(order)), [TR.get(c, c) for c in order], rotation=30, ha="right")
ax[1].set_yticks(range(len(CLASSES)), [TR.get(c, c) for c in CLASSES])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax[1].text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax[1].set_xlabel("Tahmin"); ax[1].set_ylabel("Gerçek"); ax[1].set_title(f"Test · son model + emin değilim (eşik {threshold})")
plt.tight_layout(); plt.savefig(FIG / "emin_degilim.png", dpi=150, bbox_inches="tight"); plt.show()

pred_plain = final.predict(test.text_norm, test)
inv_err = (pred_plain == "invoice") & (test.label.values != "invoice")
print(f"'Fatura' diye yapılan yanlış tahmin: {int(inv_err.sum())} → emin değilim eklenince {int((inv_err & (labels != UNSURE)).sum())}")

In [ ]:
joblib.dump(final, MODELS / "final_model.joblib")
info = {"model": "Linear SVM (kalibre)", "özellik": "karakter TF-IDF + tablo", "C": C_best, "tablo_agirligi": w_best,
        "emin_degilim_esigi": threshold, "siniflar": list(final.classes_),
        "test_accuracy": float(accuracy_score(test.label, pred_plain)),
        "test_macro_f1": float(f1_score(test.label, pred_plain, average="macro")),
        "test_emin_degilim_orani": float((labels == UNSURE).mean()),
        "test_karar_verilenlerde_dogruluk": float((labels[labels != UNSURE] == test.label.values[labels != UNSURE]).mean())}
(MODELS / "final_model.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))
print(json.dumps(info, ensure_ascii=False, indent=2))

## C. İçerik çıkarımı: önce / sonra

09'da elle değerlendirilen **aynı 20 belge** (110 alan) yeni kurallarla tekrar çıkarılır. Eski değer, eski karar ve yeni değer yan yana gösterilir.
Değişen alanlar elle yeniden işaretlenir; değişmeyenler eski kararını korur.

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/extract.py
"""Belge türüne özel içerik çıkarımı (kural tabanlı + spaCy NER).

Girdi: OCR'dan gelen temizlenmiş metin (text_clean.clean_text çıktısı) ve tahmin edilen tür.
Çıktı: {"alanlar": {...}, "ozet": "Bu bir CV: ..."}
"""
import re

try:
    import spacy
    _NLP = spacy.load("en_core_web_sm")
except Exception:  # spaCy yoksa sadece kurallar çalışır
    _NLP = None

# ---------------------------------------------------------------- ortak kalıplar
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
PHONE = re.compile(r"(?:\+?\d{1,3}[\s.-]?)?(?:\(\d{2,4}\)|\d{2,4})[\s.-]?\d{3}[\s.-]?\d{2,4}(?:[\s.-]?\d{2})?")
# v2 (21 Ekim, elle değerlendirme): belge numarası (50145702) ve yayın sayfa aralığı (30502-30509) telefon sanılıyordu.
# Artık ya "tel/phone" etiketi ya da telefon biçimi (alan kodu + 3 + 4 hane, ayraçlı) gerekiyor.
PHONE_LABELED = re.compile(r"\b(?:tel|telephone|phone|mobile|cell|gsm)\b\.?\s*(?:no\.?|number)?\s*[:#.]?\s*(\+?[\d(][\d\s().-]{6,18}\d)", re.I)
PHONE_FORMATTED = re.compile(r"(?<![\d-])(?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)\s?|\d{3}[\s.-])\d{3}[\s.-]\d{4}(?![\d-])")


def find_phone(text):
    m = PHONE_LABELED.search(text)
    if m and sum(c.isdigit() for c in m.group(1)) >= 7:
        return " ".join(m.group(1).split())
    m = PHONE_FORMATTED.search(text)
    return m.group(0).strip() if m else None
URL = re.compile(r"\b(?:https?://|www\.)\S+|\b(?:github|linkedin)\.com/\S+", re.I)
MONTHS = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?"
DATE = re.compile(
    rf"\b(?:\d{{1,2}}[/.-]\d{{1,2}}[/.-]\d{{2,4}}|{MONTHS}\s+\d{{1,2}},?\s+(?:19|20)\d{{2}}|"
    rf"\d{{1,2}}[\s.-]+{MONTHS}[\s.,-]*(?:19|20)?\d{{2}}|\d{{4}}-\d{{2}}-\d{{2}})\b",   # v2: "10.JULY.2014"
    re.I)


def find_date(text):
    """İlk geçerli tarih. v2: "70.00.00" gibi gün/ay aralığı dışındaki sayı grupları tarih sayılmaz."""
    for m in DATE.finditer(text or ""):
        d = m.group(0).strip()
        parts = re.split(r"[/.-]", d)
        if all(p.isdigit() for p in parts) and len(parts) == 3 and len(parts[0]) <= 2:
            a, b = int(parts[0]), int(parts[1])
            if not ((1 <= a <= 12 and 1 <= b <= 31) or (1 <= a <= 31 and 1 <= b <= 12)):
                continue
        return d
    return None


# v2: Avrupa yazımı (100.000,00) da tutar; "100.000,00" artık "100.00" diye kesilmiyor
_NUM = r"(?:\d{1,3}(?:\.\d{3})+(?:,\d{2})?|\d{1,3}(?:,\d{3})+(?:\.\d{2})?|\d+(?:[.,]\d{2})?)"
_NUM_DEC = r"(?:\d{1,3}(?:[.,]\d{3})+[.,]\d{2}|\d+[.,]\d{2})"   # OCR ayraçları karıştırabiliyor: 20.000.00
_CUR = r"(?:usd|eur|gbp|tl|try)\b"
MONEY = re.compile(rf"(?<![\d.,])(?:[$€£₺]\s?{_NUM}(?:\s?{_CUR})?|{_NUM_DEC}(?:\s?{_CUR})?|{_NUM}\s?{_CUR})(?![\d]|[.,]\d)", re.I)

DEGREE_WORDS = ("bachelor", "master", "phd", "ph.d", "b.s", "b.sc", "m.s", "m.sc", "b.a", "m.a", "mba",
                "degree", "engineering", "science", "lisans", "mühendisliği", "muhendisligi")
SCHOOL_WORDS = ("university", "universit", "college", "institute", "school", "academy", "üniversite", "universite")
SECTION_HEADS = ("education", "experience", "work experience", "employment", "skills", "technical skills",
                 "projects", "references", "publications", "certifications", "languages", "interests",
                 "objective", "summary", "awards", "activities")


def _lines(text):
    return [l.strip() for l in text.splitlines() if l.strip()]


def _ents(text, labels):
    if _NLP is None or not text:
        return []
    doc = _NLP(text[:20000])
    out = []
    for e in doc.ents:
        t = " ".join(e.text.split())
        if e.label_ in labels and len(t) > 1 and t not in out:
            out.append(t)
    return out


def _first(pattern, text, group=0):
    m = pattern.search(text)
    return m.group(group).strip() if m else None


def _section(lines, names):
    """Başlığı `names` içinde olan bölümün satırlarını döndürür (bir sonraki başlığa kadar)."""
    out, inside = [], False
    for l in lines:
        key = re.sub(r"[^a-z ]", "", l.lower()).strip()
        is_head = key in SECTION_HEADS and len(l) < 40
        # v2: form tipi CV'lerde başlık açıklamayla birlikte gelir: "EDUCATION/TRAINING (Begin with ...)"
        m = _LONG_HEAD.match(l)
        if not is_head and m:
            is_head, key = True, m.group(1).lower()
        if is_head:
            if inside:
                break
            inside = key in names
            continue
        if inside:
            out.append(l)
    return out


_LONG_HEAD = re.compile(r"^\s*(EDUCATION|EXPERIENCE|EMPLOYMENT|PUBLICATIONS|SKILLS)\b(?:\s*/\s*[A-Z]+)?\s*[(:]")

_NOT_NAME = SECTION_HEADS + SCHOOL_WORDS + DEGREE_WORDS + ("resume", "curriculum", "vitae", "cv", "page",
                                                           "street", "avenue", "road", "phone", "email", "address",
                                                           # v2: CV başlıkları ve form alanları ad sanılıyordu
                                                           "biographical", "sketch", "personal", "data", "principal",
                                                           "investigator", "program", "director", "grant", "number",
                                                           "position", "title", "research", "associate", "professor",
                                                           "senior", "department", "chief", "staff", "form", "birth")
def _is_name_token(t):
    """"Sena" · "ÖZKAN" · "M." · "Jean-Paul" (Türkçe harfler dahil)."""
    if len(t) == 2 and t[0].isalpha() and t[0].isupper() and t[1] == ".":
        return True
    core = t.replace("-", "")
    return len(t) >= 2 and core.isalpha() and t[0].isupper() and (t[1:].replace("-", "").islower() or t.isupper())
_NAME_SUFFIX = re.compile(r"^(?:M\.?D\.?|Ph\.?D\.?|Jr\.?|Sr\.?)$")
_NAME_LABEL = re.compile(r"^\s*(?:full\s+)?name\b\s*[:;]?\s*(.*)$", re.I)
_NOT_NAME_EXTRA = ("institution", "location", "field", "study", "year", "conferred", "birthdate", "assistant",
                   "fellow", "scientist", "lecturer", "instructor", "manager", "engineer", "head", "member")


def _name_candidate(segment):
    """Satır parçasının başındaki ad-soyadı döndürür: büyük harfle başlayan 2-4 kelime (baş harf "M." olabilir).
    Başlık/kurum kelimesine gelince durur: "Cecilia M. Giachelli Research Associate" → "Cecilia M. Giachelli"."""
    toks, out = segment.strip().split(), []
    for t in toks[:6]:
        core = t.rstrip(",")
        if not _is_name_token(core) or core.lower().strip(".") in _NOT_NAME + _NOT_NAME_EXTRA:
            break
        out.append(core)
        if t.endswith(",") or len(out) == 4:
            break
    if sum(len(w.strip(".")) >= 2 for w in out) < 2:
        return None
    name = " ".join(out)
    rest = toks[len(out):]
    if rest and _NAME_SUFFIX.match(rest[0].strip(",")):     # "Glenn R. Cunningham, M.D."
        name += ", " + rest[0].strip(",")
    return name


def _resume_name(lines):
    # 1) "Name" / "Full Name" etiketi: aynı satırda ya da sonraki 3 satırda
    for i, l in enumerate(lines[:20]):
        m = _NAME_LABEL.match(l)
        if not m:
            continue
        for cand in [m.group(1)] + lines[i + 1:i + 4]:
            for seg in re.split(r"\s*\|\s*|\s{2,}", cand or ""):
                n = _name_candidate(seg)
                if n:
                    return n
    # 2) üst satırlarda satır başında ad (form tipi CV: "Glenn R. Cunningham, M.D. VA Medical Center ...")
    for l in lines[:12]:
        n = _name_candidate(re.split(r"\s*\|\s*", l)[0])
        if n:
            return n
    # 3) eski yöntem: spaCy kişi adı ya da ad gibi görünen satır
    persons = [p for p in _ents("\n".join(lines[:8]), {"PERSON"}) if _looks_like_name(p)]
    return persons[0] if persons else next((l for l in lines[:6] if _looks_like_name(l)), None)


# açık derece: noktalı kısaltmalar (B.A., Ph.D., BS.) ya da "BA in ..."; "MA. 02138" gibi eyalet kodları sayılmaz
_STRONG_DEGREE = re.compile(r"\b(?:B\.\s?A\.?|B\.?\s?S\.|B\.\s?Sc\.?|M\.\s?S\.?|M\.\s?Sc\.?|M\.\s?A\.|Ph\.?\s?D\.?|M\.\s?D\.|"
                            r"BSc|MSc|MBA|(?:BA|BS|MS|MA)(?=\s+in\s))|\b(?:bachelor|master|doctorate)\b")
_YEAR = re.compile(r"\b(?:19|20)\d{2}\b")


def _looks_like_name(line):
    """2-4 kelimelik, harflerden oluşan, büyük harfle başlayan, başlık/okul/adres olmayan satır."""
    if any(ch.isdigit() for ch in line) or any(ch in line for ch in "@:/|()"):
        return False
    words = line.replace(",", " ").replace(".", " ").split()
    if not 2 <= len(words) <= 4 or not all(w.isalpha() for w in words) or sum(len(w) >= 2 for w in words) < 2:
        return False
    low = line.lower()
    if any(k in low.split() or k in low for k in _NOT_NAME if len(k) > 3) or low in _NOT_NAME:
        return False
    return all(w[0].isupper() for w in words)


# ---------------------------------------------------------------- türe özel çıkarıcılar
def extract_resume(text):
    lines = _lines(text)
    name = _resume_name(lines)

    edu = _section(lines, {"education"}) or lines
    schools = [l for l in edu if any(w in l.lower() for w in SCHOOL_WORDS)]
    # v2: eğitim tablosu satırını (yıl ya da derece içeren okul satırı) tercih et; unvan satırındaki kurumu değil
    school = next((l for l in schools if _YEAR.search(l) or _STRONG_DEGREE.search(l)), schools[0] if schools else None)
    if school is None:
        orgs = [o for o in _ents("\n".join(edu), {"ORG"}) if any(w in o.lower() for w in SCHOOL_WORDS)]
        school = orgs[0] if orgs else None
    # v2: önce açık derece kısaltması (B.A., Ph.D. ...) olan satır; sonra genel kelimeler (uzun açıklama ve tablo başlığı hariç)
    name_key = name.split(",")[0] if name else None
    degree = next((l for l in edu if _STRONG_DEGREE.search(l) and l != school
                   and not (name_key and name_key in l)), None)   # ad satırındaki "M.D." derece satırı değil
    if degree is None and school and _STRONG_DEGREE.search(school):   # "BA in Biochemistry (1978, University of Dublin)"
        degree = school
    if degree is None:
        degree = next((l for l in edu if any(w in l.lower() for w in DEGREE_WORDS) and l != school
                       and len(l.split()) <= 15 and not re.search(r"\binstitution\b|\byear\b", l, re.I)), None)
    if school:
        school = re.sub(r"^\s*education\s*[:;-]\s*", "", school, flags=re.I) or None

    skills_lines = _section(lines, {"skills", "technical skills"})
    skills = None
    if skills_lines:
        items = re.split(r"[,;•·|]|\s-\s", " ".join(skills_lines[:4]))
        skills = [s.strip(" :.") for s in items if 1 < len(s.strip()) < 30][:10] or None

    return {
        "ad_soyad": name,
        "universite": school,
        "bolum_derece": degree,
        "e_posta": _first(EMAIL, text),
        "telefon": find_phone(text),
        "beceriler": skills,
    }


_NOT_COMPANY = ("invoice", "quantity", "description", "price", "amount", "total", "date", "bill to",
                "ship to", "page", "tax", "qty", "item")
_CORP = re.compile(r"\b(?:inc|corp|corporation|co|company|ltd|limited|llc|plc|group|gmbh|a\.?ş|ltd\.?\s*şti)\b\.?", re.I)


def _invoice_company(lines):
    top = [l for l in lines[:12] if not any(w in l.lower() for w in _NOT_COMPANY) and re.search(r"[A-Za-z]{3}", l)]
    corp = next((l for l in top if _CORP.search(l)), None)
    if corp:
        return corp
    orgs = [o for o in _ents("\n".join(top), {"ORG"}) if not any(w in o.lower() for w in _NOT_COMPANY)]
    if orgs:
        return orgs[0]
    # antetli kâğıt: ilk 3 satırdan büyük harf ağırlıklı olan
    def upper_ratio(l):
        letters = [c for c in l if c.isalpha()]
        return sum(c.isupper() for c in letters) / len(letters) if letters else 0
    return next((l for l in top[:3] if upper_ratio(l) > 0.6 and len(l) > 4), None)


def _invoice_amounts(lines):
    """v2: tablo satırlarında miktar ve birim fiyat da para gibi görünür (10.00, 50.00). Her satırın en sağdaki
    tutarı (TUTAR sütunu) alınır; belgede en az iki "$"lı tutar varsa sadece onlar tutulur."""
    vals = []
    for l in lines:
        found = [m.strip().rstrip(",.") for m in MONEY.findall(l)]
        if found:
            vals.append(found[-1])
    dollars = [v for v in vals if v[:1] in "$€£₺"]
    vals = dollars if len(dollars) >= 2 else vals
    out = list(dict.fromkeys(vals))
    return out[:5] or None


def extract_invoice(text):
    text = re.sub(r"(\d)[,.]\s+(\d{3})\b", r"\1,\2", text)   # OCR: '1, 250.00' -> '1,250.00'
    lines = _lines(text)
    inv_no = _first(re.compile(r"invoice\s*(?:no\.?|n0\.?|number|num|#)\s*[:.]?\s*((?=[A-Z0-9/-]*\d)[A-Z0-9][A-Z0-9/-]{1,20})", re.I), text, 1)
    # v2: önce kesin toplam ifadeleri (net amount, total due ...), sonra genel "total", en son "amount"
    total = None
    for pat in (r"(?:net\s+amount|total\s+amount\s+due|amount\s+due|balance\s+due|grand\s+total|total\s+due|invoice\s+total|amount\s+payable)",
                r"\btotal", r"\bamount"):
        found = []
        for m in re.finditer(pat + r"\b[^\n$€£\d]{0,20}", text, re.I):
            mm = MONEY.match(text, m.end()) or MONEY.search(text[m.end():m.end() + 25])
            if mm:
                found.append(mm.group(0).strip().rstrip(",."))
        if found:
            total = found[-1]
            break
    company = _invoice_company(lines)
    return {
        "fatura_no": inv_no,
        "tarih": _header(text, "date|invoice\\s+date|dated") or find_date(text),
        "firma": company,
        "toplam_tutar": total,
        "tutarlar": _invoice_amounts(lines),
    }


_LABEL_ONLY = re.compile(r"^\s*(?:from|to|cc|ce|co|bcc|subject|subj|re|date|sent|sender|recipient)\s*[:;]\s*$", re.I)


def _header(text, names):
    # v2: değer aynı satırda olmalı ([ \t]*); eskiden \s* satır sonunu da atladığı için "From:" satırının
    # değeri bir alt satırdaki "Date:" sanılıyordu. Değer başka bir etiketse kabul edilmez.
    pat = re.compile(rf"^[ \t]*(?:{names})[ \t]*[:;][ \t]*(\S[^\n]*)$", re.I | re.M)
    for m in pat.finditer(text):
        val = m.group(1).strip()
        if not _LABEL_ONLY.match(val):
            return val
    return None


# OCR "Cc:" etiketini sık sık "Ce:" ya da "co:" diye okuyor
_EMAIL_LABELS = {"from": "from", "sender": "from", "to": "to", "recipient": "to", "cc": "cc", "ce": "cc", "co": "cc",
                 "bcc": "cc", "subject": "subject", "subj": "subject", "re": "subject", "date": "date", "sent": "date"}
_HEAD_PAT = re.compile(r"^\s*(?:from|to|cc|ce|co|bcc|subject|subj|re|date|sent|sender|recipient)\s*[:;]", re.I)
_WEEKDAY_TIME = re.compile(r"\b(?:mon|tue|wed|thu|fri|sat|sun)[a-z]*\b.*\d{1,2}:\d{2}|"
                           r"\b\d{1,2}:\d{2}\s*(?:am|pm)\b", re.I)


def _label_block(lines):
    """OCR etiket sütununu (From/Date/Subject/To/Cc) değerlerden ayrı bir blok olarak okuduğunda
    etiketleri sırayla değerlerle eşleştirir. Hizalama tarih satırına göre yapılır."""
    for i in range(len(lines)):
        run = []
        j = i
        while j < len(lines) and _LABEL_ONLY.match(lines[j]):
            run.append(_EMAIL_LABELS[re.match(r"\s*([a-z]+)", lines[j], re.I).group(1).lower()])
            j += 1
        if len(run) < 2:
            continue
        rest = lines[j:j + len(run) + 6]
        if "date" in run:
            d = next((k for k, l in enumerate(rest) if DATE.search(l) or _WEEKDAY_TIME.search(l)), None)
            start = d - run.index("date") if d is not None else None
        else:
            start = None
        if start is None or start < 0:
            start = next((k for k, l in enumerate(rest) if not re.fullmatch(r"[\d\s.-]+", l)), 0)
        vals = rest[start:start + len(run)]
        if len(vals) < len(run):
            return {}, None
        out = {}
        for lab, v in zip(run, vals):
            if lab == "subject":
                v = re.sub(r"^\s*re\s*[:;]\s*", "RE: ", v, flags=re.I)
            out.setdefault(lab, v.strip())
        return out, j + start + len(run)        # gövdenin başladığı satır
    return {}, None


def extract_email(text):
    lines = _lines(text)
    # 1) etiket bloğu + değer bloğu (iki sütunlu başlık)
    block, body_start = _label_block(lines)
    sender, to = block.get("from"), block.get("to")
    subject, date = block.get("subject"), block.get("date")
    # 2) etiket ve değer aynı satırda
    sender = sender or _header(text, "from|sender")
    to = to or _header(text, "to|recipient")
    subject = subject or _header(text, "subject|subj|re")
    date = date or _header(text, "date|sent")
    # 3) etiketler hiç okunmamış (Outlook çıktısı): gönderen, gün+saat satırı, alıcılar, "Subject:"
    if not sender:
        subj_i = next((k for k, l in enumerate(lines[:15]) if re.match(r"\s*(?:subject|subj)\s*[:;]", l, re.I)), None)
        d_i = next((k for k, l in enumerate(lines[:6]) if _WEEKDAY_TIME.search(l) and DATE.search(l)), None)
        if d_i is not None and d_i >= 1 and (subj_i is None or d_i < subj_i):
            sender = lines[d_i - 1].strip(" '‘’")
            date = date or lines[d_i]
            if not to and subj_i is not None and subj_i > d_i + 1:
                to = " ".join(l.strip(" '‘’") for l in lines[d_i + 1:subj_i])[:200]
            body_start = max(body_start or 0, (subj_i + 1) if subj_i is not None else d_i + 1)
    date = date or find_date(text)
    # gövde: başlık satırlarından sonra
    if body_start is None:
        head_idx = [k for k, l in enumerate(lines[:20]) if _HEAD_PAT.match(l)]
        body_start = head_idx[-1] + 1 if head_idx else 0
    body = [l for l in lines[body_start:] if not _HEAD_PAT.match(l)]
    return {
        "gonderen": sender,
        "alici": to,
        "konu": subject,
        "tarih": date,
        "e_postalar": EMAIL.findall(text)[:5] or None,
        "govde_ozeti": " ".join(body[:3])[:200] or None,
    }


def extract_form(text):
    lines = _lines(text)
    title = next((l for l in lines[:8] if l.isupper() and len(l.split()) >= 2), lines[0] if lines else None)
    pairs = {}
    for l in lines:
        m = re.match(r"^([A-Za-z][A-Za-z ./&()'-]{1,40}?)\s*[:;]\s*(.+)$", l)
        if m:
            key, val = m.group(1).strip(), m.group(2).strip(" _.")
            if val and key.lower() not in pairs and len(val) < 120:
                pairs[key.lower()] = val
        if len(pairs) >= 10:
            break
    return {
        "baslik": title,
        "alanlar": pairs or None,
        "tarih": find_date(text),
    }


_CLOSING = re.compile(r"^\s*(?:sincerely|yours\s+(?:truly|sincerely|faithfully|very\s+truly)|very\s+truly\s+yours|"
                      r"respectfully(?:\s+yours)?|best\s+regards|kind\s+regards|regards|cordially)\b", re.I)
_SALUTATION = re.compile(r"^\s*dear\s+(.{2,60}?)\s*[:,;]?\s*$", re.I | re.M)


def extract_letter(text):
    """Mektup / resmi yazı (dilekçe): kurum (antet), tarih, alıcı, hitap, konu, imza sahibi."""
    lines = _lines(text)
    salute = _first(_SALUTATION, text, 1)
    sal_idx = next((i for i, l in enumerate(lines) if _SALUTATION.match(l)), None)
    # imza: kapanış ifadesinden sonraki ilk isim gibi görünen satır
    signer = None
    close_idx = next((i for i, l in enumerate(lines) if _CLOSING.match(l)), None)
    if close_idx is not None:
        signer = next((l for l in lines[close_idx + 1:close_idx + 6] if _looks_like_name(l)), None)
        if signer is None:
            persons = _ents("\n".join(lines[close_idx + 1:close_idx + 6]), {"PERSON"})
            signer = persons[0] if persons else None
    # alıcı adresi: tarih ile "Dear" arasındaki blokta isim gibi görünen ilk satır
    head = [l for l in lines[:4] if not DATE.search(l)]
    org = next((l for l in head if _CORP.search(l)), None)
    if org is None:
        orgs = _ents("\n".join(head), {"ORG"})
        org = orgs[0] if orgs else None
    recipient = None
    if sal_idx:
        date_idx = next((i for i, l in enumerate(lines[:sal_idx]) if DATE.search(l)), -1)
        window = lines[max(date_idx + 1, sal_idx - 8):sal_idx]
        recipient = next((l for l in window if _looks_like_name(l) and l != org), None)
    return {
        "kurum": org,
        "tarih": find_date(text),
        "alici": recipient,
        "hitap": salute,
        "konu": _header(text, "re|subject|subj"),
        "imza": signer,
    }


_DATELINE = re.compile(r"^([A-Z][A-Z .,'-]{2,40})\s*(?:\((?:AP|UPI|Reuters|AFP)\))?\s*[-—–]{1,2}\s", re.M)


def extract_news(text):
    """Gazete haberi: başlık, yer (dateline), tarih, haberde geçen kişi ve kurumlar."""
    lines = _lines(text)
    title = next((l for l in lines[:6] if 3 <= len(l.split()) <= 15 and sum(c.isalpha() for c in l) > 10), None)
    return {
        "baslik": title,
        "yer": _first(_DATELINE, text, 1),
        "tarih": find_date(text),
        "kisiler": _ents(text, {"PERSON"})[:4] or None,
        "kurumlar": _ents(text, {"ORG"})[:4] or None,
    }


_NOT_TITLE = re.compile(r"\b(?:printed\s+in|journal|vol\.?|volume|pp\.|fax|copyright|received|accepted|reprint(?:ed)?|"
                        r"sonderdruck|page|inc\.?|wiley|elsevier|press)\b|©|\(c\)", re.I)
_AFFIL = re.compile(r"\b(?:university|universit|college|institute|department|dept|school|hospital|laborator|"
                    r"centre|center|company|corporation|inc)\b", re.I)
_VOCAB = None


def _vocab():
    """İngilizce kelime listesi (SymSpell sıklık sözlüğü); yoksa boş küme → sözlük kontrolü atlanır."""
    global _VOCAB
    if _VOCAB is None:
        try:
            from importlib import resources
            path = resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt"
            _VOCAB = {l.split()[0] for l in path.read_text().splitlines() if l.strip()}
        except Exception:
            _VOCAB = set()
    return _VOCAB


def _wordlike_ratio(line):
    words = re.findall(r"[A-Za-z]{2,}", line)
    if not words:
        return 0.0
    vocab = _vocab()
    if not vocab:   # sözlük yoksa: sesli harf içeren, makul uzunlukta kelime oranı
        return sum(bool(re.search(r"[aeiouy]", w, re.I)) and len(w) <= 15 for w in words) / len(words)
    return sum(w.lower() in vocab for w in words) / len(words)


def _is_title_line(l):
    words = l.split()
    letters = sum(c.isalpha() for c in l)
    alpha_words = re.findall(r"[A-Za-z]+", l)
    mean_len = sum(map(len, alpha_words)) / max(1, len(alpha_words))   # OCR çöpü kısa parçalardan oluşur
    return (len(words) >= 3 and letters >= 15 and letters / max(1, len(l.replace(" ", ""))) >= 0.85
            and mean_len >= 4 and not _NOT_TITLE.search(l) and not _AFFIL.search(l) and _wordlike_ratio(l) >= 0.7)


def _looks_like_author_line(l):
    """"Pauline M. Tang, Judith Finkelstein" gibi: virgülle ayrılmış, büyük harfle başlayan ad grupları."""
    parts = [p.strip() for p in re.split(r",|\band\b", l) if p.strip()]
    return len(parts) >= 2 and sum(bool(re.match(r"^[A-Z][\w.'-]*(?:\s+[A-Z][\w.'-]*){1,3}$", p)) for p in parts) >= 2


def _scientific_title(lines):
    """v2: dergi künyesi, faks başlığı, "Printed in Great Britain" gibi satırları atlar; başlık iki satıra
    bölünmüşse birleştirir."""
    for i, l in enumerate(lines[:15]):
        if _is_title_line(l) and len(l.split()) >= 4:
            title, j = l, i + 1
            while j < min(i + 3, len(lines)) and _is_title_line(lines[j]) and not re.match(r"^\s*by\b", lines[j], re.I) \
                    and not _looks_like_author_line(lines[j]) and not title.rstrip().endswith((".", ":")):
                title += " " + lines[j]
                j += 1
            prev = lines[i - 1] if i > 0 else ""
            if prev and prev.isupper() and l.isupper() and 1 <= len(prev.split()) <= 6 and not _NOT_TITLE.search(prev) \
                    and sum(c.isalpha() for c in prev) >= 5 and _wordlike_ratio(prev) >= 0.7:
                title = prev + " " + title           # "CHARACTERIZATION" + "OF AIRBORNE PARTICULATE MATTER"
            return title.strip(), j
    return None, 0


def _scientific_authors(lines, after):
    """Başlıktan sonraki 3 satırda: "BY ..." satırı ya da virgül/"and" ile ayrılmış adlar (kurum satırı değil)."""
    for l in lines[after:after + 3]:
        if re.fullmatch(r"[\d\s.,()-]+", l) or _AFFIL.search(l):
            continue
        if re.search(r"[\"*]\s*$", l) or re.match(r"^\W*(?:[IVX]+\.|\d+\.|\.)\s", l):   # alt başlık: "I. Vapor Phase Analysis*"
            continue
        m = re.match(r"^\s*by\s+(.+)$", l, re.I)
        cand = m.group(1) if m else l
        cand = re.split(r",\s*(?:PH|M\.?\s?D|Ph\.?\s?D|F\.?A\.?C\.?P)\b", cand)[0]   # unvanları at
        if m or re.search(r",|\band\b", cand):
            parts = [p.strip(" .") for p in re.split(r",|\band\b", cand) if len(p.strip(" .")) > 1]
            if parts and sum(bool(re.search(r"[A-Za-z]{2,}", p)) for p in parts) >= max(1, len(parts) // 2):
                return parts[:5]
        caps = [w for w in cand.split() if w[:1].isupper() or re.fullmatch(r"\d\.", w)]   # OCR: "J." → "3."
        stop = {"in", "of", "the", "for", "on", "at", "to", "from", "with", "by"}
        if 2 <= len(cand.split()) <= 5 and len(caps) == len(cand.split()) \
                and not any(w.lower() in stop for w in cand.split()):   # tek yazar: "DAVID J. LEWIS"
            return [cand.strip()]
    return None


def extract_scientific(text):
    """Bilimsel makale: başlık, yazarlar, özet (Abstract), anahtar kelimeler."""
    lines = _lines(text)
    title, after = _scientific_title(lines)
    authors = _scientific_authors(lines, after) if title else None
    if title is None:   # eski yöntem
        title = next((l for l in lines[:6] if len(l.split()) >= 4 and sum(c.isalpha() for c in l) > 15), None)
    if authors is None and title is None:
        authors = [p for p in _ents("\n".join(lines[:12]), {"PERSON"})][:5] or None
    m = re.search(r"\babstract\b[\s:.-]*(.{20,600}?)(?:\n\s*\n|\b(?:introduction|key\s*words?|keywords)\b|$)",
                  text, re.I | re.S)
    abstract = " ".join(m.group(1).split())[:300] if m else None
    kw = _first(re.compile(r"\bkey\s*words?\s*[:.-]\s*(.+)", re.I), text, 1)
    return {
        "baslik": title,
        "yazarlar": authors,
        "ozet": abstract,
        "anahtar_kelimeler": kw,
        "kaynakca_var": bool(re.search(r"\b(?:references|bibliography|literature cited)\b", text, re.I)),
    }


EXTRACTORS = {"resume": extract_resume, "invoice": extract_invoice, "email": extract_email, "form": extract_form,
              "letter": extract_letter, "news_article": extract_news, "scientific_publication": extract_scientific}
TR_NAMES = {"resume": "CV", "invoice": "fatura", "email": "e-posta", "form": "form",
            "letter": "resmi yazı / mektup", "news_article": "gazete haberi", "scientific_publication": "bilimsel makale"}


def summarize(label, fields):
    f = fields
    if label == "resume":
        parts = [f.get("ad_soyad"), f.get("universite"),
                 f.get("bolum_derece") if f.get("bolum_derece") != f.get("universite") else None]
        if f.get("beceriler"):
            parts.append("Beceriler: " + ", ".join(f["beceriler"][:5]))
    elif label == "invoice":
        parts = [f"Fatura no {f['fatura_no']}" if f.get("fatura_no") else None, f.get("firma"), f.get("tarih"),
                 f"Toplam {f['toplam_tutar']}" if f.get("toplam_tutar") else None]
    elif label == "email":
        who = " → ".join(x for x in (f.get("gonderen"), f.get("alici")) if x) or None
        parts = [who, f"Konu: {f['konu']}" if f.get("konu") else None, f.get("tarih")]
    elif label == "letter":
        parts = [f.get("kurum"), f.get("tarih"), f"Alıcı: {f.get('hitap') or f.get('alici')}" if (f.get("hitap") or f.get("alici")) else None,
                 f"Konu: {f['konu']}" if f.get("konu") else None, f"İmza: {f['imza']}" if f.get("imza") else None]
    elif label == "news_article":
        parts = [f.get("baslik"), f.get("yer"), f.get("tarih"),
                 "Kişiler: " + ", ".join(f["kisiler"][:3]) if f.get("kisiler") else None]
    elif label == "scientific_publication":
        parts = [f.get("baslik"), "Yazarlar: " + ", ".join(f["yazarlar"][:3]) if f.get("yazarlar") else None,
                 f"Anahtar kelimeler: {f['anahtar_kelimeler']}" if f.get("anahtar_kelimeler") else None]
    else:
        kv = list((f.get("alanlar") or {}).items())[:4]
        parts = [f.get("baslik")] + [f"{k} = {v}" for k, v in kv]
    parts = [p for p in parts if p]
    return f"Bu bir {TR_NAMES.get(label, label)}" + (": " + " · ".join(parts) if parts else " (içerik alanları okunamadı)")


def extract(text, label):
    fields = EXTRACTORS.get(label, extract_form)(text or "")
    return {"alanlar": fields, "ozet": summarize(label, fields)}


# ---------------------------------------------------------------- konu özeti (danışmanın "ikinci aşama" hedefi)
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

_GENERIC = {"new", "please", "thank", "thanks", "dear", "regards", "sincerely", "page", "shows", "also",
            "would", "could", "like", "just", "know", "let", "use", "used", "make", "need", "want", "good",
            "subject", "cc", "bcc", "sent", "fax", "tel", "phone", "date", "re", "fwd", "attached"}
_STOP = sorted(ENGLISH_STOP_WORDS | _GENERIC)

_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")


class TopicSummarizer:
    """Belgenin ne hakkında olduğunu anlatan anahtar kelimeler + 2-3 cümlelik çıkarımsal özet.

    Kelime ağırlıkları (TF-IDF) eğitim belgelerinden öğrenilir: bir belgede sık geçen ama
    tüm belgelerde yaygın olmayan kelimeler o belgenin konusunu temsil eder. OCR çöpü genelde
    tek bir belgede geçtiği için min_df=2 ile sözlüğe girmez.
    """

    def __init__(self, min_df=2, max_df=0.5):
        self.vec = TfidfVectorizer(stop_words=_STOP, ngram_range=(1, 2), min_df=min_df, max_df=max_df,
                                   sublinear_tf=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b")

    def fit(self, texts):
        self.vec.fit([t for t in texts if isinstance(t, str)])
        self.vocab_ = self.vec.vocabulary_
        self.names_ = self.vec.get_feature_names_out()
        return self

    def keywords(self, text, k=6):
        if not text:
            return []
        row = self.vec.transform([text]).toarray().ravel()
        order = np.argsort(row)[::-1]
        out = []
        for i in order:
            if row[i] <= 0 or len(out) >= k:
                break
            term = self.names_[i]
            if any(term in o or o in term for o in out):   # "invoice" ile "invoice number" tekrarını önle
                continue
            out.append(term)
        return out

    def _sentences(self, text):
        sents = []
        for s in _SENT_SPLIT.split(text):
            s = " ".join(s.split())
            words = s.split()
            alpha = sum(w.isalpha() for w in words)
            if len(words) >= 5 and alpha / len(words) >= 0.6 and len(s) <= 300:
                sents.append(s)
        return sents

    def summary(self, text, n=2):
        sents = self._sentences(text or "")
        if not sents:
            return []
        m = self.vec.transform(sents)
        scores = np.asarray(m.sum(axis=1)).ravel() / np.sqrt(np.maximum(m.getnnz(axis=1), 1))
        top = sorted(np.argsort(scores)[::-1][:n])
        return [sents[i] for i in top if scores[i] > 0]

    def describe(self, text, label, k=6, n=2):
        kws = self.keywords(text, k)
        sents = self.summary(text, n)
        tur = TR_NAMES.get(label, label)
        line = f"Bu bir {tur}."
        if kws:
            line += f" Konusu / anahtar kelimeler: {', '.join(kws)}."
        return {"anahtar_kelimeler": kws, "ozet_cumleler": sents, "konu_ozeti": line}


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/text_clean.py
import re
import unicodedata

# OCR'ın sık ürettiği tipografik karakterleri sadeleştir
_CHAR_MAP = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "•": "-", "·": "-",
    " ": " ", "­": "",
})
_WORDISH = re.compile(r"[A-Za-z0-9]")
_NOISE_TOKEN = re.compile(r"^[^\w$%@&#+]+$")          # sadece sembol: | ~ ` ' . , _ vb.
_EDGE_JUNK_L = re.compile(r"^[|_~`'\"\s]+")
_EDGE_JUNK_R = re.compile(r"[|_~`\s]+$")
_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
_URL = re.compile(r"\b(?:https?://|www\.)\S+", re.I)
_MONEY = re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?")
_NUM = re.compile(r"\d+")


def alnum_ratio(s):
    s = s.replace(" ", "")
    return sum(c.isalnum() for c in s) / len(s) if s else 0.0


def clean_text(raw, min_line_alnum=0.4):
    """OCR metnini temizler; büyük/küçük harf, e-posta, tutar gibi bilgiler korunur
    (bilgi çıkarımı aşamasında gerekecek)."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).translate(_CHAR_MAP)
    t = "".join(c for c in t if c == "\n" or c.isprintable())
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)               # satır sonu tirelemesini birleştir
    lines = []
    for line in t.split("\n"):
        tokens = [tok for tok in line.split() if not _NOISE_TOKEN.match(tok)]
        line = " ".join(tokens)
        line = _EDGE_JUNK_R.sub("", _EDGE_JUNK_L.sub("", line))
        if len(line) < 2 or not _WORDISH.search(line):
            continue
        if alnum_ratio(line) < min_line_alnum:              # çoğu sembol olan çöp satır
            continue
        lines.append(line)
    return "\n".join(lines)


def normalize_for_model(clean):
    """Sınıflandırıcı için: küçük harf + e-posta/URL/tutar/sayıları ortak simgeye çevir.
    Böylece model 'john@acme.com' yerine 'bir e-posta adresi var' bilgisini öğrenir."""
    t = clean.lower()
    t = _EMAIL.sub(" emailaddr ", t)
    t = _URL.sub(" urladdr ", t)
    t = _MONEY.sub(" moneyamt ", t)
    t = _NUM.sub("0", t)
    t = re.sub(r"[^a-z0\s]", " ", t)
    return " ".join(t.split())


def garbage_ratio(raw):
    """Ham OCR çıktısındaki anlamsız token oranı (harf/rakam içermeyen ya da
    harf-rakam-sembol karışık kısa parçalar)."""
    toks = raw.split() if isinstance(raw, str) else []
    if not toks:
        return 1.0
    bad = 0
    for tok in toks:
        if not _WORDISH.search(tok):
            bad += 1
        elif len(tok) <= 3 and alnum_ratio(tok) < 0.67:
            bad += 1
    return bad / len(toks)


In [ ]:
import extract; importlib.reload(extract)
EVAL_ROWS = [{"belge": "images/test/email/test_email_00143.png", "tur": "email", "alan": "gonderen", "deger": "Ostern, Eric on Thu, Apr 4, 1996 10:45 AM", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00143.png", "tur": "email", "alan": "alici", "deger": "Portnoy, Sharon", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00143.png", "tur": "email", "alan": "konu", "deger": "FW: NYC Rest. Contacts", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00143.png", "tur": "email", "alan": "tarih", "deger": "Apr 4, 1996", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00143.png", "tur": "email", "alan": "e_postalar", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00143.png", "tur": "email", "alan": "govde_ozeti", "deger": "Ce: Bayliss, Elissa; Culley, Liz; Young, Tracy Hi Sharon: Thad Elissa follow-up on obtaining info from the restaurants b", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00124.png", "tur": "email", "alan": "gonderen", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/email/test_email_00124.png", "tur": "email", "alan": "alici", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/email/test_email_00124.png", "tur": "email", "alan": "konu", "deger": "Washington Post Story", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00124.png", "tur": "email", "alan": "tarih", "deger": "April 29, 1998", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00124.png", "tur": "email", "alan": "e_postalar", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00124.png", "tur": "email", "alan": "govde_ozeti", "deger": "Carothers, Paul Wednesday, April 29, 1998 6:07 PM Knuth, Kathy S. (MS Mail); Dennis, Darienne; Ostronic, John;", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00159.png", "tur": "email", "alan": "gonderen", "deger": "ebh@caunxi.ctd.crnl.gov (B4 Harris)", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00159.png", "tur": "email", "alan": "alici", "deger": "cvrGdsunix?.derd.crnl.gov", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00159.png", "tur": "email", "alan": "konu", "deger": "Tobacco Funding Question", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00159.png", "tur": "email", "alan": "tarih", "deger": "Tue, 26 Mar 1996 17:40:32 -0500", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00159.png", "tur": "email", "alan": "e_postalar", "deger": "ebh@caunxi.ctd.crnl.gov, sikkavk@dsunix2.dsrd.ornl.cov, _vks@ornl.gov", "karar": "Doğru"}, {"belge": "images/test/email/test_email_00159.png", "tur": "email", "alan": "govde_ozeti", "deger": "Ed_Harris,3/26/96 4:40 PM,Tobacco Funding Question ce: sikkavk@dsunix2.dsrd.ornl.cov vee,", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00178.png", "tur": "email", "alan": "gonderen", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00178.png", "tur": "email", "alan": "alici", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00178.png", "tur": "email", "alan": "konu", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00178.png", "tur": "email", "alan": "tarih", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00178.png", "tur": "email", "alan": "e_postalar", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00178.png", "tur": "email", "alan": "govde_ozeti", "deger": "52021 9188 yO 8 Buunjoeynuey spuesa \"YO \"BuURDeINUEN SUSHPOUIERD MUAANG", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00100.png", "tur": "email", "alan": "gonderen", "deger": "Date:", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00100.png", "tur": "email", "alan": "alici", "deger": "co:", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00100.png", "tur": "email", "alan": "konu", "deger": "To:", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00100.png", "tur": "email", "alan": "tarih", "deger": "Subject:", "karar": "Yanlış"}, {"belge": "images/test/email/test_email_00100.png", "tur": "email", "alan": "e_postalar", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/email/test_email_00100.png", "tur": "email", "alan": "govde_ozeti", "deger": "co: 2071557599 Note for Knuff, Ketrina", "karar": "Yanlış"}, {"belge": "images/test/invoice/test_invoice_00263.png", "tur": "invoice", "alan": "fatura_no", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/invoice/test_invoice_00263.png", "tur": "invoice", "alan": "tarih", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00263.png", "tur": "invoice", "alan": "firma", "deger": "PHILIP MORRIS MANAGEMENT CORPORATION", "karar": "Yanlış"}, {"belge": "images/test/invoice/test_invoice_00263.png", "tur": "invoice", "alan": "toplam_tutar", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00263.png", "tur": "invoice", "alan": "tutarlar", "deger": "1723.58", "karar": "Yanlış"}, {"belge": "images/test/invoice/test_invoice_00241.png", "tur": "invoice", "alan": "fatura_no", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00241.png", "tur": "invoice", "alan": "tarih", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00241.png", "tur": "invoice", "alan": "firma", "deger": "WoRsN", "karar": "Yanlış"}, {"belge": "images/test/invoice/test_invoice_00241.png", "tur": "invoice", "alan": "toplam_tutar", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00241.png", "tur": "invoice", "alan": "tutarlar", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00296.png", "tur": "invoice", "alan": "fatura_no", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/invoice/test_invoice_00296.png", "tur": "invoice", "alan": "tarih", "deger": "1/24/89", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00296.png", "tur": "invoice", "alan": "firma", "deger": "PROMOTIONAL MARKETING, INC", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00296.png", "tur": "invoice", "alan": "toplam_tutar", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/invoice/test_invoice_00296.png", "tur": "invoice", "alan": "tutarlar", "deger": "$25,500, $17,400,, $37,950", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00218.png", "tur": "invoice", "alan": "fatura_no", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/invoice/test_invoice_00218.png", "tur": "invoice", "alan": "tarih", "deger": "02/18/94", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00218.png", "tur": "invoice", "alan": "firma", "deger": "Brand Precielon Services, Inc. 1so0a761 it", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00218.png", "tur": "invoice", "alan": "toplam_tutar", "deger": "1,130.00", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00218.png", "tur": "invoice", "alan": "tutarlar", "deger": "10.00 , 50.00 , 500.00 , 10.00 , 22.00", "karar": "Yanlış"}, {"belge": "images/test/invoice/test_invoice_00298.png", "tur": "invoice", "alan": "fatura_no", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/invoice/test_invoice_00298.png", "tur": "invoice", "alan": "tarih", "deger": "November 8 00", "karar": "Yanlış"}, {"belge": "images/test/invoice/test_invoice_00298.png", "tur": "invoice", "alan": "firma", "deger": "ANYNEX Company", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00298.png", "tur": "invoice", "alan": "toplam_tutar", "deger": "$956.83", "karar": "Doğru"}, {"belge": "images/test/invoice/test_invoice_00298.png", "tur": "invoice", "alan": "tutarlar", "deger": "$956.83, 9638.67 , 5638.67 , $5.10, 313.06", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00341.png", "tur": "resume", "alan": "ad_soyad", "deger": "BIOGRAPHICAL SKETCH", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00341.png", "tur": "resume", "alan": "universite", "deger": "Tel-Aviv University, Tel-Aviv, Israel B.Sc. [1966-1969 Biology", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00341.png", "tur": "resume", "alan": "bolum_derece", "deger": "Tel-Aviv University, Tel-Aviv, Israel Ph.D. [1973-1977 Endocrinology of Reproductic:", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00341.png", "tur": "resume", "alan": "e_posta", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00341.png", "tur": "resume", "alan": "telefon", "deger": "30502-30509", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00341.png", "tur": "resume", "alan": "beceriler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00323.png", "tur": "resume", "alan": "ad_soyad", "deger": "EDWARD DIALA", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00323.png", "tur": "resume", "alan": "universite", "deger": "Harvard University Special Student in Biology 1972-4", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00323.png", "tur": "resume", "alan": "bolum_derece", "deger": "Institution Subject & Degree Year", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00323.png", "tur": "resume", "alan": "e_posta", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00323.png", "tur": "resume", "alan": "telefon", "deger": "50145702", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00323.png", "tur": "resume", "alan": "beceriler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00356.png", "tur": "resume", "alan": "ad_soyad", "deger": "ANN KATHERINE DALY", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00356.png", "tur": "resume", "alan": "universite", "deger": "BA in Biochemistry (1978, University of Dublin); PhD in Biochemistry (1982,", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00356.png", "tur": "resume", "alan": "bolum_derece", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/resume/test_resume_00356.png", "tur": "resume", "alan": "e_posta", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00356.png", "tur": "resume", "alan": "telefon", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00356.png", "tur": "resume", "alan": "beceriler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00374.png", "tur": "resume", "alan": "ad_soyad", "deger": "Cecilia M. Giachelli Research", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00374.png", "tur": "resume", "alan": "universite", "deger": "University of Washington, Seattle, WA Phb, 1987 Pharmacolo", "karar": "Doğru"}, {"belge": "images/test/resume/test_resume_00374.png", "tur": "resume", "alan": "bolum_derece", "deger": "American Association for the Advancement of Science", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00374.png", "tur": "resume", "alan": "e_posta", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00374.png", "tur": "resume", "alan": "telefon", "deger": "50304562", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00374.png", "tur": "resume", "alan": "beceriler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00301.png", "tur": "resume", "alan": "ad_soyad", "deger": "Pose Bs", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00301.png", "tur": "resume", "alan": "universite", "deger": "Metabolism, Baylor College of Medicine", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00301.png", "tur": "resume", "alan": "bolum_derece", "deger": "Key personnel typical wil include all Indviduls wih doctoral or other professional degrees, btn some projects wil includ", "karar": "Yanlış"}, {"belge": "images/test/resume/test_resume_00301.png", "tur": "resume", "alan": "e_posta", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00301.png", "tur": "resume", "alan": "telefon", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/resume/test_resume_00301.png", "tur": "resume", "alan": "beceriler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00046.png", "tur": "scientific_publication", "alan": "baslik", "deger": "Printed i Great Brin", "karar": "Yanlış"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00046.png", "tur": "scientific_publication", "alan": "yazarlar", "deger": "DAVID 3", "karar": "Yanlış"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00046.png", "tur": "scientific_publication", "alan": "ozet", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00046.png", "tur": "scientific_publication", "alan": "anahtar_kelimeler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00046.png", "tur": "scientific_publication", "alan": "kaynakca_var", "deger": "False", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00057.png", "tur": "scientific_publication", "alan": "baslik", "deger": "What the Physician Should Know About Tobacco", "karar": "Doğru"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00057.png", "tur": "scientific_publication", "alan": "yazarlar", "deger": "WALTER A. BASTEDO, tobaccos", "karar": "Doğru"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00057.png", "tur": "scientific_publication", "alan": "ozet", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00057.png", "tur": "scientific_publication", "alan": "anahtar_kelimeler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00057.png", "tur": "scientific_publication", "alan": "kaynakca_var", "deger": "False", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00030.png", "tur": "scientific_publication", "alan": "baslik", "deger": "Sondereudh us: eitge to Tabklordhng\" Band 8- Heb 7 Ohtake 976", "karar": "Yanlış"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00030.png", "tur": "scientific_publication", "alan": "yazarlar", "deger": "Cytrel, Dav Git Vidroy", "karar": "Yanlış"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00030.png", "tur": "scientific_publication", "alan": "ozet", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00030.png", "tur": "scientific_publication", "alan": "anahtar_kelimeler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00030.png", "tur": "scientific_publication", "alan": "kaynakca_var", "deger": "False", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00012.png", "tur": "scientific_publication", "alan": "baslik", "deger": "PN LEE, w+ FIR RED Ntel cit Gees", "karar": "Yanlış"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00012.png", "tur": "scientific_publication", "alan": "yazarlar", "deger": "LEE, GP Se, Sretrcewn, Wie Lis", "karar": "Yanlış"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00012.png", "tur": "scientific_publication", "alan": "ozet", "deger": "", "karar": "Kaçırıldı"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00012.png", "tur": "scientific_publication", "alan": "anahtar_kelimeler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00012.png", "tur": "scientific_publication", "alan": "kaynakca_var", "deger": "False", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00051.png", "tur": "scientific_publication", "alan": "baslik", "deger": "TOBACCO SHOKE AND THE PULMONARY ALVEOLAR MACROPHAGE", "karar": "Doğru"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00051.png", "tur": "scientific_publication", "alan": "yazarlar", "deger": "DRATH, P. DAVIES, M. L. KARNOVSKY, Beth Israel Hospital Boston, Teceresrtct", "karar": "Doğru"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00051.png", "tur": "scientific_publication", "alan": "ozet", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00051.png", "tur": "scientific_publication", "alan": "anahtar_kelimeler", "deger": "", "karar": "Belgede yok"}, {"belge": "images/test/scientific_publication/test_scientific_publication_00051.png", "tur": "scientific_publication", "alan": "kaynakca_var", "deger": "False", "karar": "Belgede yok"}]
ev = pd.DataFrame(EVAL_ROWS)
docs = list(dict.fromkeys(ev.belge))
ocr = pd.concat([pd.read_csv(DATA / f) for f in ["ocr_dataset.csv", "ocr_dataset_aday.csv"] if (DATA / f).exists()])
ocr = ocr.drop_duplicates("path").set_index("path")
import text_clean, ast; importlib.reload(text_clean)

def norm(v):
    """Eski (CSV'deki metin) ve yeni değerleri aynı biçime getirir: liste → "a, b"; boş → bulunamadı."""
    if isinstance(v, str):
        t = v.strip()
        if t in ("", "None", "nan", "[]"):
            return "— (bulunamadı)"
        if t.startswith("[") and t.endswith("]"):
            try: v = ast.literal_eval(t)
            except Exception: return " ".join(t.split())
        else:
            return " ".join(t.split())
    if v is None or v == [] or v == "":
        return "— (bulunamadı)"
    if isinstance(v, list):
        return ", ".join(" ".join(str(x).split()) for x in v)
    return " ".join(str(v).split())

new_vals = {}
for b in docs:
    tur = ev[ev.belge == b].tur.iloc[0]
    f = extract.extract(text_clean.clean_text(ocr.loc[b, "text"]), tur)["alanlar"]
    for alan in ev[ev.belge == b].alan:
        new_vals[(b, alan)] = norm(f.get(alan))
ev["eski_deger"] = ev.deger.map(norm)
ev["yeni_deger"] = [new_vals[(b, a)] for b, a in zip(ev.belge, ev.alan)]
ev["degisti"] = ev.eski_deger != ev.yeni_deger
print(f"{int(ev.degisti.sum())} / {len(ev)} alanın değeri değişti")
pd.set_option("display.max_colwidth", 70)
ev["belge_no"] = ev.belge.map({b: i + 1 for i, b in enumerate(docs)})
display(ev[ev.degisti][["belge_no", "tur", "alan", "eski_deger", "karar", "yeni_deger"]])
print("DEGISENLER_JSON" + ev[ev.degisti][["belge_no", "alan", "eski_deger", "karar", "yeni_deger"]].to_json(orient="records", force_ascii=False))
ev.to_csv(ROOT / "cikarim_once_sonra.csv", index=False)

### Dönük sayfa düzeltme

Elle değerlendirmede belge 7 (yaklaşık 77° dönük fatura) hiç okunamamıştı. Demoya (05) yön düzeltme eklendi:
yazı satırlarının açısı −90°…+90° arasında aranır (satır toplamlarının en keskin olduğu açı), sayfa o açıyla ve ters yönüyle okunur;
açı küçükse Tesseract'ın yön tespiti (OSD) baş aşağı sayfaya bakar. En çok ve en emin kelimeyi veren hâl seçilir; asıl hâlden
en az %20 iyi değilse sayfaya dokunulmaz. Aynı 20 belgede kontrol: düz belgeler bozulmamalı, dönük belgeler okunmalı.

In [ ]:
from ocr_core import ocr_with_orientation, run_ocr
rows_r = []
for k, b in enumerate(docs, 1):
    img = cv2.imread(str(DATA / b), cv2.IMREAD_GRAYSCALE)
    before = run_ocr(img)
    _, after, ang = ocr_with_orientation(img, upscale=False)
    rows_r.append({"belge_no": k, "tur": TR.get(ev[ev.belge == b].tur.iloc[0]), "dondurme": ang,
                   "kelime_once": before["n_words"], "kelime_sonra": after["n_words"],
                   "guven_once": round(before["mean_conf"], 1), "guven_sonra": round(after["mean_conf"], 1)})
    if ang:
        tur = ev[ev.belge == b].tur.iloc[0]
        f = extract.extract(text_clean.clean_text(after["text"]), tur)["alanlar"]
        print(f"Belge {k} {ang}° döndürüldü → yeni alanlar:", {a: norm(v) for a, v in f.items()})
        print("OCR metni (ilk satırlar):\n" + "\n".join(after["text"].splitlines()[:8]))
rot = pd.DataFrame(rows_r)
display(rot)
rot.to_csv(ROOT / "yon_duzeltme_20belge.csv", index=False)
print("Döndürülen belge sayısı:", int((rot.dondurme > 0).sum()))